# ⏰  Day 1: The First Morning

Welcome to Morning 0. This notebook is your "Hello World" for recursive thinking in Apache Spark 4.1+ (or DBR 17.0+). Before we tackle massive shuffles, we need to understand how to drive the loop.

### Roadmap:
* **1. Standard CTE:** A quick baseline of flat, non-recursive logic.
* **2. The Baby Counter:** A pure syntax test of the `ANCHOR` and `RECURSIVE` mechanics.
* **3. Folder Tree Walk:** Traversing a hierarchy of unknown depth while using tracking arrays to prevent infinite loops.
* **4. Depth Profiler:** The golden rule—measuring row growth per iteration to kill data explosions early.

### Initialize your Spark session (Ensure Spark 4.1.0+ or DBR 17.0+)

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
print(f"Current Spark Version: {spark.version}")

Current Spark Version: 4.1.2


### CASE 1: Standard vs. Nested Subqueries (The Common Table Expression)

In [2]:
print("\n--- Case 1: Testing a standard, flat Common Table Expression ---")

# Mocking local flat data
sales_data = [(1, 101, 150), (2, 102, 0), (3, 101, 50)]
spark.createDataFrame(sales_data, ["order_id", "customer_id", "amount"]).createOrReplaceTempView("raw_sales")

# Running a clean, readable flat CTE
# Note: Timer wraps `.show()` because Spark is lazy; action triggers execution.
spark.sql("""
WITH clean_sales AS (
  SELECT order_id, customer_id, amount
  FROM raw_sales
  WHERE amount > 0
)
SELECT customer_id, sum(amount) AS total_amount
FROM clean_sales
GROUP BY customer_id
""").show()


--- Case 1: Testing a standard, flat Common Table Expression ---
+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|        101|         200|
+-----------+------------+



### CASE 2: The Baby Groundhog (A Simple Recursive Counter)
This minimal example showcases the pure mechanics of recursion
without any complex dataframes, joins, or shuffles.

In [3]:
print("\n--- Case 2: Running a basic recursive counter ---")

spark.sql("""
WITH RECURSIVE counter(n) AS (
  -- Anchor Member: Seed the loop with the initial state
  VALUES (1)

  UNION ALL

  -- Recursive Member: Grab the previous number and add 1
  SELECT n + 1
  FROM counter
  WHERE n < 5 -- Loop termination criteria
)
SELECT * FROM counter
""").show()


--- Case 2: Running a basic recursive counter ---
+---+
|  n|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
+---+



### CASE 3: Traversing a Tree of Unknown Depth
Let's mock up a nested folder directory and walk through it recursively.

In [4]:
print("\n--- Case 3: Preparing hierarchical data and running tree traversal ---")

folders_data = [
    (1, None, "root"),
    (2, 1, "finance"),
    (3, 1, "engineering"),
    (4, 2, "reports"),
    (5, 4, "2026"),
    (6, 5, "final_really")
]

folders_df = spark.createDataFrame(folders_data, ["id", "parent_id", "name"])
folders_df.createOrReplaceTempView("folders")

# The Recursive Query:
# We add a 'depth' column to track iterations and a 'path' array to spot circular references.
tree_df = spark.sql("""
WITH RECURSIVE folder_tree AS (
  -- Anchor Query: Start exactly at the root level (where parent_id is NULL)
  SELECT
    id,
    parent_id,
    name,
    0 AS depth,
    array(id) AS path
  FROM folders
  WHERE parent_id IS NULL

  UNION ALL

  -- Recursive Step: Join base folders against yesterday's discovered paths
  SELECT
    f.id,
    f.parent_id,
    f.name,
    ft.depth + 1 AS depth,
    concat(ft.path, array(f.id)) AS path
  FROM folders f
  JOIN folder_tree ft ON f.parent_id = ft.id
  -- Operational Safeguard: Prevent infinite loops if data has cycles
  WHERE NOT array_contains(ft.path, f.id)
)
SELECT * FROM folder_tree ORDER BY path
""")

tree_df.show(truncate=False)


--- Case 3: Preparing hierarchical data and running tree traversal ---
+---+---------+------------+-----+---------------+
|id |parent_id|name        |depth|path           |
+---+---------+------------+-----+---------------+
|1  |NULL     |root        |0    |[1]            |
|2  |1        |finance     |1    |[1, 2]         |
|4  |2        |reports     |2    |[1, 2, 4]      |
|5  |4        |2026        |3    |[1, 2, 4, 5]   |
|6  |5        |final_really|4    |[1, 2, 4, 5, 6]|
|3  |1        |engineering |1    |[1, 3]         |
+---+---------+------------+-----+---------------+



### CASE 4: The Golden Rule - Profile Your Recursion Depths
Always check your row counts per depth level to catch runaway data explosions early.

In [5]:
print("\n--- Case 4: Profiling row production per depth iteration ---")

spark.sql("""
WITH RECURSIVE folder_tree AS (
  SELECT id, parent_id, name, 0 AS depth, array(id) AS path
  FROM folders WHERE parent_id IS NULL
  UNION ALL
  SELECT f.id, f.parent_id, f.name, ft.depth + 1 AS depth, concat(ft.path, array(f.id)) AS path
  FROM folders f JOIN folder_tree ft ON f.parent_id = ft.id
  WHERE NOT array_contains(ft.path, f.id)
)
SELECT
  depth,
  count(*) AS rows_produced
FROM folder_tree
GROUP BY depth
ORDER BY depth
""").show()


--- Case 4: Profiling row production per depth iteration ---
+-----+-------------+
|depth|rows_produced|
+-----+-------------+
|    0|            1|
|    1|            2|
|    2|            1|
|    3|            1|
|    4|            1|
+-----+-------------+



## 📊 Post-Lab Analysis: The 3-Minute Plot Twist

We got the exact results we wanted under **Spark 4.1.2**, but the execution times just handed us a massive technical cliffhanger.

### 🛑 The Shocking Numbers
* **Case 1 (Flat CTE):** ~21 seconds (Standard cold-start overhead)
* **Case 2 (Baby Counter):** ~0.9 seconds (Fast, local scalar math)
* **Case 3 & 4 (Tree Traversal & Profiler):** **~3 minutes 16 seconds each**

> 💡 **Processing a 6-row table took over 3 minutes.** A native Python loop does this in under 2 milliseconds. Why?

### 🧠 Why the Groundhog is Moving in Slow Motion
You just hit the **Micro-Data Tax** of distributed engines running iterative loops:

1. **The Catalyst Planning Loop:** A recursive CTE isn't planned just once. For every single depth level (0 → 4), Spark's physical loop engine (`UnionLoopExec`) has to pause, check for new rows, compile a *brand-new* physical graph, and schedule new tasks.
2. **Distributed Freight Train Overhead:** Spark is built to haul gigabytes across clusters. The coordination energy required to manage task boundaries, system state, and metadata for *6 measly records* across 5 distinct loop iterations means framework overhead eats 99.9% of your compute time.